# Drawing inductors in Klayout

This Notebook draws inductors in `klayout` instead of `gdspy`.

In [1]:
import klayout.db as db
import numpy as np

if "sg13g2" not in db.Technology.technology_names():
    tech = db.Technology()
    tech.load("../lib/ihp_open_pdk/ihp-sg13g2/libs.tech/klayout/tech/sg13g2.lyt")
    db.Technology.register_technology(tech)
else: 
    tech = db.Technology.technology_by_name("sg13g2")

In [2]:

layout = db.Layout()
layout.technology_name = "sg13g2"
top = layout.create_cell("TOP")

inductor = layout.create_cell("Inductor")

# Layer stackup
signal_layer = layout.layer(134, 0)     # TopMetal2
underpass_layer = layout.layer(126, 0)  # TopMetal1

signal_pin_layer = layout.layer(layout.get_info(signal_layer).layer, 2)
signal_underpass_via_layer = layout.layer(layout.get_info(signal_layer).layer - 1, 0)

ind_layer = layout.layer(27,0)
ind_pin_layer = layout.layer(27,2)
ind_text_layer = layout.layer(27,25)

no_fill_layers = [
    layout.layer(1, 23),   # Activ.nofill
    layout.layer(5, 23),   # GatPoly.nofill
    layout.layer(8, 23),   # Metal1.nofill
    layout.layer(10, 23),  # Metal2.nofill
    layout.layer(30, 23),  # Metal3.nofill
    layout.layer(50, 23),  # Metal4.nofill
    layout.layer(67, 23),  # Metal5.nofill
    layout.layer(126, 23), # TopMetal1.nofill
    layout.layer(134, 23), # TopMetal2.nofill
]

no_rcx_layer = layout.layer(148, 0)

print(tech.default_grid())

0.005


In [3]:
def roundToGrid(tech, num, floor=False, ceil=False):
    """
    Round the number to technology grid.
    If the technology is not defined, or the grid is 0, prints the warning message and returns the same number.
    """
    if tech==None:
        print("WARNING: roundToGrid called, but techonology is not defined")
        return num
    grid = tech.default_grid()
    if grid==0.0:
        print("WARNING: roundToGrid called, but techonology grid=0.0")
        return num
    #print(f"Rounding {num} to grid of size {grid}")
    if floor or ceil:
        return np.floor(num/grid)*grid if floor else np.ceil(num/grid)*grid
    else:
        return np.round(num/grid)*grid

In [4]:
w = 10.98        # microns
s = 2            # microns
r = 125          # Outter radius
N = 5            # Number of turns

conn_len = 3*w   # Length of connection legs

via_width = 0.9
via_spacing = 1.06
via_pitch = via_width + via_spacing
via_enc = 0.5

#(https://github.com/dgrujic/pcLab/issues/2)
e=roundToGrid(tech, w+s+w/(1+np.sqrt(2)) + 2*s*np.tan(np.pi / 8)) 


In [5]:
def make45Bridge(w, l, offsetX, offsetY, addVias=False): 
    """
    w : float
        metal width
    l : float
        bridge length
    """

    # Start at the origin
    x, y = roundToGrid(tech,-(2*w+s)/2), roundToGrid(tech,-l/2)
    p1 = db.DPoint(x, y)
    # Move to the corner of the first trace
    x += w
    p2 = db.DPoint(x, y)
    # Extend this side of the first trace to match DRC
    y += roundToGrid(tech, l/2 - offsetX/2 - (w/2)/(1 + np.sqrt(2))) + offsetY
    p3 = db.DPoint(x, y)
    # 45 degree turn
    x += offsetX
    y += offsetX
    p4 = db.DPoint(x, y)
    # Reach the second trace
    y += roundToGrid(tech, l/2 - offsetX/2 + (w/2)/(1 + np.sqrt(2))) + offsetY
    p5 = db.DPoint(x, y)
    # Move to the corner of the second trace
    x -= w
    p6 = db.DPoint(x, y)
    # Extend this side of the second trace to maintain width at w
    y -= roundToGrid(tech, l/2 - offsetX/2 - (w/2)/(1 + np.sqrt(2))) + offsetY
    p7 = db.DPoint(x, y)
    # Go back to origin
    x -= offsetX
    y -= offsetX
    p8 = db.DPoint(x, y)

    # Add connection pads for vias. Connection pads are square shaped and have a side equal to w
    if addVias:
        p1.y -= w
        p2.y -= w
        p5.y += w
        p6.y += w

    ## TODO: Implement fillVias function
    polygon = db.DPolygon([p1, p2, p3, p4, p5, p6, p7, p8])
    return polygon

In [6]:
def octFill(r) :
    a = r/(1 + np.sqrt(2))  # Half a side of the octagon
    p1 = db.DPoint(-a,-r)
    p2 = db.DPoint(a,-r)
    p3 = db.DPoint(r,-a)
    p4 = db.DPoint(r,a)
    p5 = db.DPoint(a, r)
    p6 = db.DPoint(-a, r)
    p7 = db.DPoint(-r,a)
    p8 = db.DPoint(-r,-a)

    polygon = db.DPolygon([p1, p2, p3, p4, p5, p6, p7, p8])
    return polygon


def octSegment(w, r, e):
    """
    w : float
        metal width
    r : float
        outer radius
    e : float
        spacing for bridges
    """
    ## NOTE: `a` is `side/sqrt(2)`
    a=roundToGrid(tech, (2*r)/(np.sqrt(2)+2))               #  length of outer corner
    r=roundToGrid(tech, r)
    # Inner corner length must be floored instead of rounded
    c=roundToGrid(tech, (2*r)/(np.sqrt(2)+2)+w/(np.sqrt(2)+1), floor=True) #  length of inner corner

    # Start from the outside and leave space for bridges
    x = r
    y = e/2
    p1 = db.DPoint(x,y)
    # Move upwards (vertical segment)
    y = r - a
    p2 = db.DPoint(x,y)
    # 45 degree turn
    x -= a
    y += a
    p3 = db.DPoint(x,y)
    # Get to the end of the horizontal segment
    x = e/2
    p4 = db.DPoint(x,y)
    # Move downwards to begin the inner trace
    y -= w
    p5 = db.DPoint(x,y)
    # Complete the horizontal segment
    x = r - c
    p6 = db.DPoint(x,y)
    # 45 degree turn
    x = r - w
    y = r - c
    p7 = db.DPoint(x,y)
    # Complete the vertical segment
    y = e/2
    p8 = db.DPoint(x,y)

    polygon = db.DPolygon([p1, p2, p3, p4, p5, p6, p7, p8])
    return polygon


In [7]:
# Draw the segments
for n in range(N):
    seg = octSegment(w, r - n *(w + s), e)
    for i in range(4):
        seg.transform(db.DTrans(rot=45*i))
        inductor.shapes(signal_layer).insert(seg)

# Draw underpasses and bridges
via_cell = layout.create_cell("VIA")
via_cell.shapes(signal_underpass_via_layer).insert(db.DBox(0,0,via_width,via_width))
for n in range(N-1):
    bridge = make45Bridge(w, e, w+s, 0, addVias=True)
    bridge.transform(db.Trans(rot=45))
    bridge.transform(db.Trans( 0, (-1)**(2 + n)*(r - (1+n)*(w + s) + s/2) ))
    upass = bridge.transformed(db.Trans(rot=90, mirrx=True))
    inductor.shapes(signal_layer).insert(bridge)
    inductor.shapes(underpass_layer).insert(upass)
    # Add vias to the underpass
    left_bottom = min(upass.each_point_hull(), key=lambda p: (p.x, p.y))
    right_bottom = max(upass.each_point_hull(), key=lambda p: (p.x, -p.y))
    via_number = np.floor((w - 2*via_enc + via_spacing) / via_pitch)
    array_size = via_number * via_width + (via_number - 1) * via_spacing
    centering_offset = via_enc + (w - 2*via_enc - array_size)/2
    left_vias = db.DCellInstArray(
        via_cell.cell_index(),
        db.DTrans(left_bottom.x  + centering_offset, left_bottom.y + centering_offset),
        db.DVector(via_pitch,0),
        db.DVector(0,via_pitch),
        via_number,
        via_number
    )
    right_vias = db.DCellInstArray(
        via_cell.cell_index(),
        db.DTrans(right_bottom.x  - centering_offset - via_width, right_bottom.y + centering_offset),
        db.DVector(-via_pitch,0),
        db.DVector(0,via_pitch),
        via_number,
        via_number
    )

    inductor.insert(left_vias)
    inductor.insert(right_vias)


# Patch segments to form turns
for n in range(N):
    patch = db.DBox(db.DPoint(r, e/2), db.DPoint(r-w, -e/2))
    patch.move(-n*(w + s), 0)
    inductor.shapes(signal_layer).insert(patch)
    inductor.shapes(signal_layer).insert(patch.transformed(db.DTrans(rot=90)))

# Patch the inner turn
patch = db.DBox(db.DPoint(e/2, r - (N-1)*(w + s)), db.DPoint(-e/2, r-w- (N-1)*(w + s)))
if not (N % 2):
    patch = patch.transformed(db.DTrans(rot=90))
inductor.shapes(signal_layer).insert(patch)

# Draw the connections
conn = db.DBox(db.DPoint(-e/2 - w/2, -r + w), db.DPoint(-e/2 + w/2, -r - conn_len))
inductor.shapes(signal_layer).insert(conn)
inductor.shapes(signal_layer).insert(conn.moved(e, 0))

# Merge all polygons in signal layer
region = db.Region(inductor.begin_shapes_rec(signal_layer))
region.merge()
inductor.shapes(signal_layer).clear()
inductor.shapes(signal_layer).insert(region)

# Draw the pins
pin_height = s
pin = db.DBox(db.DPoint(-e/2 - w/2, -r - conn_len + pin_height), db.DPoint(-e/2 + w/2, -r - conn_len))
inductor.shapes(signal_pin_layer).insert(pin)
inductor.shapes(signal_pin_layer).insert(pin.moved(e, 0))
inductor.shapes(ind_pin_layer).insert(pin)
inductor.shapes(ind_pin_layer).insert(pin.moved(e, 0))

# Draw "No fill" regions
oct = octFill(r + conn_len)
for layer in no_fill_layers:
    inductor.shapes(layer).insert(oct)

# Prevent parasitics extraction
inductor.shapes(no_rcx_layer).insert(oct)

# Mark the region as an inductor
inductor.shapes(ind_layer).insert(oct)

# Label pins
pin1_label = db.DText("P1", -e/2 - w/2, -r - conn_len)
pin2_label = db.DText("P2", e/2 - w/2, -r - conn_len)
inductor.shapes(ind_text_layer).insert(pin1_label)
inductor.shapes(ind_text_layer).insert(pin2_label)



text ('P2',r0 4102,-157940)

In [8]:
# Put the inductor cell as a child inside TOP
ind_cell = db.CellInstArray(inductor.cell_index(), db.Trans())
top.insert(ind_cell)

ind_name = "inductor"
out_file = ".out/"+ ind_name+ ".gds"
layout.write(out_file)

In [9]:
# Create a gds2palace compatible file also
top.clear()

# Place each pin in a different layer
palace_pin1_layer = layout.layer(201, 0)
palace_pin2_layer = layout.layer(202, 0)

inductor.shapes(palace_pin1_layer).insert(pin.enlarged(0, -pin.height()/4).moved(0, -pin.height()/4))
inductor.shapes(palace_pin2_layer).insert(pin.enlarged(0, -pin.height()/4).moved(e, -pin.height()/4))

# Add a ground frame
palace_ground_layer = layout.layer(8, 0)  # Metal1

ind_bbox = inductor.dbbox()

gframe = db.DPolygon(ind_bbox.enlarged(roundToGrid(tech, r/2 + 5*w)))
gframe_hole = ind_bbox.enlarged(roundToGrid(tech, r/2))
gframe.insert_hole(gframe_hole)

gframe_pin = db.DBox(db.DPoint(pin.left - 5, pin.top),  db.DPoint(pin.right + e + 5, gframe_hole.bottom))

inductor.shapes(palace_ground_layer).insert(gframe)
inductor.shapes(palace_ground_layer).insert(gframe_pin)

# Merge all polygons in groundframe layer
region = db.Region(inductor.begin_shapes_rec(palace_ground_layer))
region.merge()
inductor.shapes(palace_ground_layer).clear()
inductor.shapes(palace_ground_layer).insert(region)

ind_cell = db.CellInstArray(inductor.cell_index(), db.Trans())
top.insert(ind_cell)
em_gds = ".out/"+ ind_name + "_forEM.gds"
layout.write(em_gds)


# `gds2palace` flow

In [ ]:
import gds2palace as gp
import os, subprocess, pathlib, sys
import ihp

ihp.PDK.activate()

In [11]:
## Add the scripts folder to path
notebook_root = pathlib.Path(os.getcwd())
repo_root     = notebook_root.parent.absolute()
scripts_root  = repo_root / pathlib.Path("scripts") 

print(str(scripts_root))
os.environ["PATH"] += os.pathsep + os.pathsep.join([str(scripts_root)])
print(os.environ["PATH"])

os.environ["DISPLAY"] = "localhost:12.0"

/home/simon/GitRepos/InductorLab/scripts
/home/simon/.pyenv/versions/InductorLab/bin:/home/simon/.pyenv/versions/3.12.12/envs/InductorLab/bin:/home/simon/.vscode-server/cli/servers/Stable-ce099c1ed25d9eb3076c11e4a280f3eb52b4fbeb/server/bin/remote-cli:/home/simon/.local/bin:/home/simon/.local/bin:/home/simon/.local/bin:/home/simon/.pyenv/plugins/pyenv-virtualenv/shims:/home/simon/.pyenv/shims:/home/simon/.pyenv/bin:/home/simon/.local/bin:/home/simon/.local/bin:/home/simon/.nix-profile/bin:/nix/var/nix/profiles/default/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin:/home/simon/GitRepos/InductorLab/scripts


In [12]:
em_gds = ".out/" + ind_name + '_forEM' + '.gds'

In [13]:
# ======================== workflow settings ================================

#start solver after creating the model?
start_simulation = False
run_command = ['./run_sim']   

In [14]:
# ===================== input files and path settings =======================

gds_filename = em_gds
XML_filename = "../resources/SG13G2_200um.xml"

# preprocess GDSII for safe handling of cutouts/holes?
preprocess_gds = True

# merge via polygons with distance less than .. microns, set to 0 to disable via merging.
merge_polygon_size = 1.5

# set and create directory for simulation output
sim_path = gp.utilities.create_sim_path(".out/", ind_name)
print('Simulation data directory: ', sim_path)

Simulation data directory:  .out/palace_model/inductor_data


In [15]:
# ======================== simulation settings ================================


settings = {}

settings['unit']   = 1e-6  # geometry is in microns
settings['margin'] = 150    # distance in microns from GDSII geometry boundary to simulation boundary 
settings['air_around'] = 50

settings['fstart']  = 0e9
settings['fstop']   = 10e9
settings['fstep']   = 0.5e9

settings["fdump"] = [2.5e9]  # save field dump at these frequency points 
# settings["fpoint"] = [15e9] # discrete frequency points, no field dump

settings['refined_cellsize'] = 5  # mesh cell size in conductor region
settings['cells_per_wavelength'] = 10   # how many mesh cells per wavelength, must be 10 or more

settings['meshsize_max'] = 70  # microns, override cells_per_wavelength 
settings['adaptive_mesh_iterations'] = 0

settings['no_gui'] = True


# Ports from GDSII Data, polygon geometry from specified special layer
# Excitations can be switched off by voltage=0, those S-parameter will be incomplete then

simulation_ports = gp.simulation_setup.all_simulation_ports()
# instead of in-plane port specified with target_layername, we here use via port specified with from_layername and to_layername
simulation_ports.add_port(gp.simulation_setup.simulation_port(portnumber=1, voltage=1, port_Z0=50, source_layernum=201, from_layername='Metal1', to_layername='TopMetal2', direction='z'))
simulation_ports.add_port(gp.simulation_setup.simulation_port(portnumber=2, voltage=1, port_Z0=50, source_layernum=202, from_layername='Metal1', to_layername='TopMetal2', direction='z'))


In [16]:
# ======================== simulation ================================

# get technology stackup data
materials_list, dielectrics_list, metals_list = gp.stackup_reader.read_substrate(XML_filename)
# get list of layers from technology
layernumbers = metals_list.getlayernumbers()
layernumbers.extend(simulation_ports.portlayers)

allpolygons = gp.gds_reader.read_gds(gds_filename, layernumbers, purposelist=[0], metals_list=metals_list, preprocess=preprocess_gds, merge_polygon_size=merge_polygon_size, cellname="Inductor")


########### create model ###########

settings['simulation_ports'] = simulation_ports
settings['materials_list'] = materials_list
settings['dielectrics_list'] = dielectrics_list
settings['metals_list'] = metals_list
settings['layernumbers'] = layernumbers
settings['allpolygons'] = allpolygons
settings['sim_path'] = sim_path 
settings['model_basename'] = ind_name 


# list of ports that are excited (set voltage to zero in port excitation to skip an excitation!)
excite_ports = simulation_ports.all_active_excitations()
config_name, data_dir = gp.simulation_setup.create_palace (excite_ports, settings)


# for convenience, write run script to model directory
gp.utilities.create_run_script(sim_path)


if start_simulation:
    try:
        os.chdir(sim_path)
        subprocess.run(run_command, shell=True)
    except:
        print(f"Unable to run Palace using command ",run_command)
    finally:
        # Go back the the original working directory
        os.chdir(notebook_root)

Reading GDSII input file: .out/inductor_forEM.gds
Pre-processing GDSII to handle cutouts and self-intersecting polygons
Using boundary condition  ['ABC', 'ABC', 'ABC', 'ABC', 'ABC', 'ABC']
Starting to create mesh file and config file
---------------------------------------------------
Wavelength in air: 30000.0 units
  meshsize_max: 70.0  units
  max_cellsize_air: 3000.0 units
---------------------------------------------------
Adding metal tags ...
Adding ports ... Union - Splitting faces                                                                                                  
Adding dielectrics ...
Dielectric =  AIR                                                                                                        
Dielectric =  Passive
Dielectric =  SiO2
Dielectric =  EPI
Dielectric =  Substrate
Dielectric =  airbox
Dielectric  AIR  with max_cellsize_local =  70 units
Dielectric  Passive  with max_cellsize_local =  70 units
Dielectric  SiO2  with max_cellsize_local =  70 

Info    : Optimization starts (volume = 1.12285e+07) with worst = 5.50497e-06 / average = 0.453819:
Info    : 0.00 < quality < 0.10 :      3833 elements
Info    : 0.10 < quality < 0.20 :      5820 elements
Info    : 0.20 < quality < 0.30 :      4902 elements
Info    : 0.30 < quality < 0.40 :      5953 elements
Info    : 0.40 < quality < 0.50 :      8066 elements
Info    : 0.50 < quality < 0.60 :      7384 elements
Info    : 0.60 < quality < 0.70 :      4506 elements
Info    : 0.70 < quality < 0.80 :      3727 elements
Info    : 0.80 < quality < 0.90 :      3904 elements
Info    : 0.90 < quality < 1.00 :      1477 elements
Info    : 2634 edge swaps, 131 node relocations (volume = 1.12285e+07): worst = 4.75297e-05 / average = 0.46485 (Wall 0.146975s, CPU 0.148091s)
Info    : 2920 edge swaps, 174 node relocations (volume = 1.12285e+07): worst = 0.000452149 / average = 0.465608 (Wall 0.263175s, CPU 0.265047s)
Info    : 2964 edge swaps, 189 node relocations (volume = 1.12285e+07): worst = 0

Info    : 1105 edge swaps, 98 node relocations (volume = 46814.9): worst = 0.00095486 / average = 0.129418 (Wall 0.122011s, CPU 0.123199s)
Info    : 0.00 < quality < 0.10 :      2442 elements
Info    : 0.10 < quality < 0.20 :      6778 elements
Info    : 0.20 < quality < 0.30 :       149 elements
Info    : 0.30 < quality < 0.40 :         4 elements
Info    : 0.40 < quality < 0.50 :         0 elements
Info    : 0.50 < quality < 0.60 :         0 elements
Info    : 0.60 < quality < 0.70 :         0 elements
Info    : 0.70 < quality < 0.80 :         0 elements
Info    : 0.80 < quality < 0.90 :         0 elements
Info    : 0.90 < quality < 1.00 :         0 elements
Info    : Optimizing volume 34
Info    : Optimization starts (volume = 1.62305e+08) with worst = 0.00011163 / average = 0.217042:
Info    : 0.00 < quality < 0.10 :       786 elements
Info    : 0.10 < quality < 0.20 :       825 elements
Info    : 0.20 < quality < 0.30 :       528 elements
Info    : 0.30 < quality < 0.40 :       35

Info    : Done writing '.out/palace_model/inductor_data/inductor.msh'
Info    : Clearing all models and views...
Info    : Done clearing all models and views


# Simulation results

In [17]:
import skrf as rf

freq = 2.5e9
omega = 2*np.pi*freq

network = rf.Network(".out/palace_model/" + ind_name + "_data/output/" + ind_name + "/" + ind_name +  ".s2p")
z11=network.z[0::,0,0]
z21=network.z[0::,1,0]
z12=network.z[0::,0,1]
z22=network.z[0::,1,1]

Zdiff = z11-z12-z21+z22
Ldiff = Zdiff.imag/omega
Rdiff = Zdiff.real
Qdiff = Zdiff.imag/Zdiff.real

findex = network.frequency.f.tolist().index(freq)
print(f"""Zdiff={Zdiff[findex]}
Ldiff={Ldiff[findex]}
Rdiff={Rdiff[findex]}
Qdiff={Qdiff[findex]}
""")

Zdiff=(8.049622488338002+100.45339972944953j)
Ldiff=6.395062046931182e-09
Rdiff=8.049622488338002
Qdiff=12.479268422212686

